# Ablation: rule-based vs Gemini start nodes for G-reasoner (`fin10k_v0`)

G-reasoner starts its search from the entities named in the question (start nodes). The main notebook
(`g_reasoner_fin10k.ipynb`) finds them with the rule-based matcher (`entity_index.py`). gfmrag's default
uses an LLM for this step, so this notebook checks whether Gemini would change the results.

**Step 1, on your Local (calls Gemini, needs `GEMINI_API_KEY` in the repo-root `.env`):**
```
cd G-reasoner
python3 gemini_ner.py --repo .. --repeat 3
```
It writes `G-reasoner/results/gemini_ner_fin10k_v0/start_nodes.jsonl` (rule vs Gemini start nodes and
timings for the 11 covered golden questions and 9 paraphrase probes) and `matching_summary.json`.
Drive syncs that file here.

**Step 2, this notebook (T4 GPU, no Gemini calls):** runs the same evaluation twice in one runtime, once
with the rule matcher and once with Gemini's saved start nodes. Graph, model, questions and metrics are
identical; only the start nodes differ. Run all cells top to bottom.


## 1. Install Python 3.12
gfmrag needs Python 3.12, so it lives in its own venv at `/content/py312`, separate from Colab's own packages.

In [1]:
!apt-get update -qq
!apt-get install -y -qq python3.12 python3.12-venv
!python3.12 --version

W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu noble InRelease' does not seem to provide it (sources.list entry misspelt?)
Selecting previously unselected package python3-pip-whl.
(Reading database ... 127332 files and directories currently installed.)
Preparing to unpack .../python3-pip-whl_24.0+dfsg-1ubuntu1.3_all.deb ...
Unpacking python3-pip-whl (24.0+dfsg-1ubuntu1.3) ...
Selecting previously unselected package python3-setuptools-whl.
Preparing to unpack .../python3-setuptools-whl_68.1.2-2ubuntu1.2_all.deb ...
Unpacking python3-setuptools-whl (68.1.2-2ubuntu1.2) ...
Selecting previously unselected package python3.12-venv.
Preparing to unpack .../python3.12-venv_3.12.3-1ubuntu0.17_amd64.deb ...
Unpacking python3.12-venv (3.12.3-1ubuntu0.17) ...
Setting up python3-setuptools-whl (68.1.2-2ubuntu1.2) ...
Setting up python3-pip-whl (24.0+dfsg-1ubuntu1.3) ...
Setting up python3.12-venv (3.12.3-1ubuntu0.17) ...
Python 3.12.

## 2. Create the venv and install gfmrag + ninja
Slow (several minutes). gfmrag is pinned to 2.0.0, the version the scripts were written against.

In [2]:
!python3.12 -m venv /content/py312
!/content/py312/bin/python -m pip install -q --upgrade pip
!/content/py312/bin/python -m pip install -q gfmrag==2.0.0 ninja
!/content/py312/bin/python -c "import gfmrag, ninja; print('gfmrag + ninja imported successfully')"

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.8/1.8 MB 18.0 MB/s eta 0:00:00
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
INFO 10-21 11:13:45 [__init__.py:216] Automatically detected platform cuda.
gfmrag + ninja imported successfully


## 3. Put the venv on PATH
gfmrag steps run with the venv's Python (`PYTHON`). PATH must include the venv so ninja is found when G-reasoner compiles its CUDA kernel on first use.

In [3]:
import os, subprocess

PYTHON = "/content/py312/bin/python"
os.environ["PATH"] = "/content/py312/bin:" + os.environ["PATH"]
print("ninja:", subprocess.run(["which", "ninja"], capture_output=True, text=True).stdout)

ninja: /content/py312/bin/ninja



## 4. Patch gfmrag's PyG version parsing
gfmrag crashes on PyG versions like `2.8.0.post1`. This keeps only the first three parts.

In [4]:
from pathlib import Path

path = Path("/content/py312/lib/python3.12/site-packages/gfmrag/models/ultra/layers.py")
text = path.read_text()
old = 'pyg_version = [int(i) for i in torch_geometric.__version__.split(".")]'
new = 'pyg_version = [int(i) for i in torch_geometric.__version__.split(".")[:3]]'
if old in text:
    path.write_text(text.replace(old, new))
    print("Patch applied.")
elif new in text:
    print("Patch already applied.")
else:
    print("Original line not found. Check layers.py manually.")

Patch applied.


## 5. Mount Google Drive and find the repo
Colab asks for permission the first time. The cell looks for the repo in the usual places (uploaded to My Drive, synced from a computer with Google Drive for desktop, or cloned into this runtime); if it finds none or several, set `REPO` at the top of the cell. gfmrag's working data and results go to `My Drive/fin_greasoner_cache/` either way.

In [6]:
import glob
from google.colab import drive
drive.mount("/content/drive")

# Where is the repo? Leave REPO = None to search the usual places, or set it yourself, e.g.
#   REPO = "/content/drive/MyDrive/financial-graphrag"   (repo folder uploaded to My Drive)
#   REPO = "/content/financial-graphrag"                 (cloned into this runtime with git)
REPO = None
if REPO is None:
    patterns = ["/content/drive/MyDrive/financial-graphrag",            # uploaded to My Drive
                "/content/drive/MyDrive/*/financial-graphrag",
                "/content/drive/MyDrive/*/*/financial-graphrag",
                "/content/drive/Othercomputers/*/*/financial-graphrag",  # Drive for desktop sync
                "/content/drive/Othercomputers/*/financial-graphrag",
                "/content/financial-graphrag"]                          # git clone in Colab
    found = sorted({p for pat in patterns for p in glob.glob(pat) if os.path.isdir(f"{p}/G-reasoner")})
    print("repo copies found:", found or "NONE")
    assert len(found) == 1, "Set REPO above to your copy of the repo (see G-reasoner/README.md, 'Setup')."
    REPO = found[0]
CODE = f"{REPO}/G-reasoner"
print("using repo:", REPO)

CACHE = "/content/drive/MyDrive/fin_greasoner_cache"  # gfmrag data root + results (survives runtime resets)
DATA  = f"{CACHE}/data"
os.makedirs(DATA, exist_ok=True)

need = [f"{CODE}/{f}" for f in ["prepare_data.py", "entity_index.py", "gfm_components.py", "index_graph.py", "query.py"]]
need += [f"{REPO}/data/graph/fin10k_v0/processed/stage1/{f}" for f in ["nodes.csv", "relations.csv", "edges.csv"]]
missing = [p.replace(REPO + "/", "") for p in need if not os.path.exists(p)]
print("missing files:", missing or "none (if some are missing, Drive may still be uploading)")

Mounted at /content/drive
synced repo: ['/content/drive/Othercomputers/My Mac/FinGraphRAG/financial-graphrag']
missing files: none (if some are missing, Drive may still be uploading)


## 6. Prepare the graph
Copies `data/graph/fin10k_v0/processed/stage1` from the clone into `DATA` and writes `raw/documents.json`, which gfmrag requires even when stage1 exists. If the team graph changed since the last run, the old embedding cache is removed.

In [7]:
!python "{CODE}/prepare_data.py" --repo "{REPO}" --graph fin10k_v0 --root "{DATA}"

unchanged stage1 -> /content/drive/MyDrive/fin_greasoner_cache/data/fin10k_v0/processed/stage1
wrote 1855 documents -> /content/drive/MyDrive/fin_greasoner_cache/data/fin10k_v0/raw/documents.json


## 7. Embed the graph (stage2)
First run: downloads Qwen3-Embedding-0.6B and embeds all 3,618 nodes (a few minutes). Later runs find the cache on Drive and finish quickly. This runs on its own so the query step starts only one vLLM model on the T4.

In [8]:
!{PYTHON} "{CODE}/index_graph.py" --root "{DATA}" --data-name fin10k_v0

INFO 10-21 11:19:15 [__init__.py:216] Automatically detected platform cuda.
vLLM limited to gpu_memory_utilization=0.35, max_model_len=8192

stage2 ready in 2s: /content/drive/MyDrive/fin_greasoner_cache/data/fin10k_v0/processed/stage2/71345f28ce99ab5be49b591c8c642cc6
nodes 3618, edges 28862 (with inverses), relations 22, feat_dim 1024
nodes by type: {'attribute': 1094, 'document': 1855, 'entity': 669}
documents loaded: 1855
files: ['graph.pt', 'node2id.json', 'rel2id.json']


## 8. Look at the two matchers' start nodes
From the file `gemini_ner.py` wrote. `same` means Gemini and the rules picked identical start nodes.

In [9]:
import json
SN = f"{CODE}/results/gemini_ner_fin10k_v0/start_nodes.jsonl"
for line in open(SN):
    r = json.loads(line)
    diff = "same" if r["agree"] else f"rules only {r['only_rule']} | Gemini only {r['only_gemini']}"
    print(f"{r['qid']:>4} [{r['set']}] Gemini {sum(r['gemini_ms'])/len(r['gemini_ms']):5.0f} ms  {diff}")
print(json.dumps(json.load(open(f"{CODE}/results/gemini_ner_fin10k_v0/matching_summary.json")), indent=2))

 q01 [golden] Gemini  1137 ms  same
 q03 [golden] Gemini  1520 ms  same
 q06 [golden] Gemini  1069 ms  same
 q08 [golden] Gemini  1438 ms  same
 q09 [golden] Gemini  1160 ms  same
 q10 [golden] Gemini  1614 ms  same
 q11 [golden] Gemini  1324 ms  same
 q14 [golden] Gemini  2057 ms  same
 q15 [golden] Gemini   995 ms  same
 q16 [golden] Gemini   910 ms  same
 q18 [golden] Gemini   963 ms  same
 p01 [probe] Gemini   940 ms  rules only ['Revenue'] | Gemini only []
 p02 [probe] Gemini  1077 ms  rules only [] | Gemini only ['FY2023 (JNJ)', 'Revenue']
 p03 [probe] Gemini  1184 ms  rules only [] | Gemini only ['FY2024 (MSFT)']
 p04 [probe] Gemini   892 ms  rules only ['Net income'] | Gemini only ['Operating income']
 p05 [probe] Gemini  1061 ms  rules only ['FY2025 (JNJ)', 'FY2025 (MSFT)', 'Revenue'] | Gemini only ['Apple Inc.']
 p06 [probe] Gemini  1028 ms  same
 p07 [probe] Gemini   947 ms  same
 p08 [probe] Gemini  1042 ms  rules only ['Ireland'] | Gemini only []
 p09 [probe] Gemini  1026 

## 9. Evaluate with the rule-based matcher (same as the main notebook's section 9)
Rerun here so both latencies come from the same runtime.

In [10]:
import time
STAMP = time.strftime('%Y%m%d-%H%M%S')
RULE_RUN = f"{CACHE}/results/ner-ablation-{STAMP}/rules"
!{PYTHON} "{CODE}/evaluate.py" --repo "{REPO}" --root "{DATA}" --data-name fin10k_v0 --out "{RULE_RUN}" 2>&1 | grep -v -E "EngineCore|^INFO|Gloo|WARNING|NCCL"

vLLM limited to gpu_memory_utilization=0.35, max_model_len=8192
`torch_dtype` is deprecated! Use `dtype` instead!
No mentioned entities found in the query. Use the query as is for entity linking.
no graph node for 'warm-up'; query runs without it
W1021 11:20:58.418000 6426 torch/utils/cpp_extension.py:2425] TORCH_CUDA_ARCH_LIST is not set, all archs for visible cards are included for compilation. 
W1021 11:20:58.418000 6426 torch/utils/cpp_extension.py:2425] If this is not desired, please set os.environ['TORCH_CUDA_ARCH_LIST'] to specific architectures.
=== G-reasoner loaded in 53s on cuda:0 ===
node types: {'attribute': 1094, 'document': 1855, 'entity': 669}
11 questions covered by the graph; skipped (gold not in graph): ['q02', 'q04', 'q05', 'q07', 'q12', 'q13', 'q17', 'q19']
Load rspmm extension. This may take a while...
  q01 [single] hit@5=1 recall=0.20  start=['Revenue', 'Apple Inc.', 'FY2025 (AAPL)']
  q03 [single] hit@5=0 recall=0.00  start=['Research and development expense', 

## 10. Evaluate with Gemini's start nodes
`evaluate.py --start-nodes` uses the saved Gemini start nodes and adds Gemini's measured time per question
to the latency (`matching_ms`, `total_ms`).

In [11]:
GEMINI_RUN = f"{CACHE}/results/ner-ablation-{STAMP}/gemini"
!{PYTHON} "{CODE}/evaluate.py" --repo "{REPO}" --root "{DATA}" --data-name fin10k_v0 --out "{GEMINI_RUN}" \
    --start-nodes "{SN}" 2>&1 | grep -v -E "EngineCore|^INFO|Gloo|WARNING|NCCL"

vLLM limited to gpu_memory_utilization=0.35, max_model_len=8192
`torch_dtype` is deprecated! Use `dtype` instead!
No mentioned entities found in the query. Use the query as is for entity linking.
no graph node for 'warm-up'; query runs without it
W1021 11:23:42.965000 7764 torch/utils/cpp_extension.py:2425] TORCH_CUDA_ARCH_LIST is not set, all archs for visible cards are included for compilation. 
W1021 11:23:42.965000 7764 torch/utils/cpp_extension.py:2425] If this is not desired, please set os.environ['TORCH_CUDA_ARCH_LIST'] to specific architectures.
=== G-reasoner loaded in 35s on cuda:0 ===
node types: {'attribute': 1094, 'document': 1855, 'entity': 669}
11 questions covered by the graph; skipped (gold not in graph): ['q02', 'q04', 'q05', 'q07', 'q12', 'q13', 'q17', 'q19']
Load rspmm extension. This may take a while...
  q01 [single] hit@5=1 recall=0.20  start=['Apple Inc.', 'FY2025 (AAPL)', 'Revenue']
  q03 [single] hit@5=0 recall=0.00  start=['FY2025 (MSFT)', 'Microsoft Corporat

## 11. Compare
Same retrieved chunks per question means the matcher made no difference to retrieval.

In [12]:
import json
def load(d):
    return [json.loads(l) for l in open(f"{d}/per_question.jsonl")], json.load(open(f"{d}/summary.json"))
(rule_rows, rule_sum), (gem_rows, gem_sum) = load(RULE_RUN), load(GEMINI_RUN)
for s in rule_sum + gem_sum:
    k = s["k"]
    print(f"{s['config']:<24} k={k:<2} Hit@{k} {s[f'Hit@{k}']:.2f}  R@{k} {s[f'R@{k}']:.2f}  "
          f"matching {s['matching_ms']:7.1f} ms  retrieval {s['retrieval_ms']:5.0f} ms  total {s['total_ms']:6.0f} ms")
same = sum(a["retrieved"] == b["retrieved"] for a, b in zip(rule_rows, gem_rows))
print(f"\nidentical retrieved lists: {same} of {len(rule_rows)} (question x k)")
print("results saved under", f"{CACHE}/results/ner-ablation-{STAMP}")

g-reasoner               k=5  Hit@5 0.18  R@5 0.04  matching     3.5 ms  retrieval  2159 ms  total   2159 ms
g-reasoner               k=10 Hit@10 0.18  R@10 0.04  matching     3.5 ms  retrieval  2159 ms  total   2159 ms
g-reasoner+gemini-ner    k=5  Hit@5 0.18  R@5 0.04  matching  1289.7 ms  retrieval   262 ms  total   1552 ms
g-reasoner+gemini-ner    k=10 Hit@10 0.18  R@10 0.04  matching  1289.7 ms  retrieval   262 ms  total   1552 ms

identical retrieved lists: 22 of 22 (question x k)
results saved under /content/drive/MyDrive/fin_greasoner_cache/results/ner-ablation-20261021-111922
